# BENGALURU HOUSE PRICE ANALYSIS

In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
df = pd.read_csv("Bengaluru_House_Data.csv")

In [3]:
df.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


In [4]:
df.shape

(13320, 9)

In [5]:
df.columns

Index(['area_type', 'availability', 'location', 'size', 'society',
       'total_sqft', 'bath', 'balcony', 'price'],
      dtype='str')

In [6]:
df.dtypes

area_type           str
availability        str
location            str
size                str
society             str
total_sqft          str
bath            float64
balcony         float64
price           float64
dtype: object

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     13320 non-null  str    
 1   availability  13320 non-null  str    
 2   location      13319 non-null  str    
 3   size          13304 non-null  str    
 4   society       7818 non-null   str    
 5   total_sqft    13320 non-null  str    
 6   bath          13247 non-null  float64
 7   balcony       12711 non-null  float64
 8   price         13320 non-null  float64
dtypes: float64(3), str(6)
memory usage: 936.7 KB


In [8]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
bath,13247.0,2.692610,1.341458,1.0,2.0,2.0,3.0,40.0
balcony,12711.0,1.584376,0.817263,0.0,1.0,2.0,2.0,3.0
price,13320.0,112.565627,148.971674,8.0,50.0,72.0,120.0,3600.0


In [9]:
# Step:1 Check missing values

In [10]:
df.isnull().sum()

area_type          0
availability       0
location           1
size              16
society         5502
total_sqft         0
bath              73
balcony          609
price              0
dtype: int64

In [11]:
(df.isnull().mean() * 100).round(2)

area_type        0.00
availability     0.00
location         0.01
size             0.12
society         41.31
total_sqft       0.00
bath             0.55
balcony          4.57
price            0.00
dtype: float64

In [12]:
# Step 2: Check duplicate records

In [13]:
df.duplicated().sum()

np.int64(529)

In [14]:
df = df.drop_duplicates()

In [15]:
df.shape

(12791, 9)

In [16]:
# Step 3: Clean categorical columns

In [17]:
# Remove extra spaces
text_columns = ["area_type", "availability", "location", "society"]

for col in text_columns:
    df[col] = df[col].str.strip()
    df[col] = df[col].str.replace(r"\s+", " ", regex=True)

"Super built-up  Area"  →  "Super built-up Area"
"Electronic City Phase II " → "Electronic City Phase II"
"  Ready To Move" → "Ready To Move"

In [18]:
# Step 4: Clean the size column and extract bedrooms

In [19]:
# Clean the size column and extract bedrooms
df["size"].value_counts(dropna=False).head(20)

size
2 BHK         4931
3 BHK         4120
4 Bedroom      824
4 BHK          574
3 Bedroom      535
1 BHK          521
2 Bedroom      314
5 Bedroom      291
6 Bedroom      191
1 Bedroom      104
8 Bedroom       84
7 Bedroom       82
5 BHK           59
9 Bedroom       46
6 BHK           30
7 BHK           17
NaN             16
1 RK            13
10 Bedroom      12
9 BHK            8
Name: count, dtype: int64

In [20]:
df["bedrooms"] = pd.to_numeric(
    df["size"].str.extract(r"(\d+)", expand=False),
    errors="coerce"
)
df["bedrooms"] = df["bedrooms"].astype("Int64")

In [21]:
df[["size", "bedrooms"]].drop_duplicates().sort_values(
    "bedrooms",
    na_position="last"
).head(20)

,size,bedrooms
20,1 BHK,1
24,1 RK,1
43,1 Bedroom,1
0,2 BHK,2
48,2 Bedroom,2
2,3 BHK,3
14,3 Bedroom,3
1,4 Bedroom,4
6,4 BHK,4
176,5 Bedroom,5


In [22]:
# Step 5: Handle missing values

In [23]:
# Handle missing location

In [24]:
df[df["location"].isnull()]

,area_type,availability,location,size,society,total_sqft,bath,balcony,price,bedrooms
568,Super built-up Area,Ready To Move,NaN,3 BHK,Grare S,1600,3.0,2.0,86.0,3


In [25]:
df = df.dropna(subset=["location"])

In [26]:
# Handle missing size and bedrooms

In [27]:
df[df["size"].isnull()]

,area_type,availability,location,size,society,total_sqft,bath,balcony,price,bedrooms
579,Plot Area,Immediate Possession,Sarjapur Road,NaN,Asiss B,1200 - 2400,NaN,NaN,34.185,<NA>
1775,Plot Area,Immediate Possession,IVC Road,NaN,Orana N,2000 - 5634,NaN,NaN,124.000,<NA>
2264,Plot Area,Immediate Possession,Banashankari,NaN,NaN,2400,NaN,NaN,460.000,<NA>
2809,Plot Area,Immediate Possession,Sarjapur Road,NaN,AsdiaAr,1200 - 2400,NaN,NaN,28.785,<NA>
2862,Plot Area,Immediate Possession,Devanahalli,NaN,Ajleyor,1500 - 2400,NaN,NaN,46.800,<NA>
5333,Plot Area,Immediate Possession,Devanahalli,NaN,Emngs S,2100 - 5405,NaN,NaN,177.115,<NA>
6423,Plot Area,Immediate Possession,Whitefield,NaN,SRniaGa,2324,NaN,NaN,26.730,<NA>
6636,Plot Area,Immediate Possession,Jigani,NaN,S2enste,1500,NaN,NaN,25.490,<NA>
6719,Plot Area,Immediate Possession,Hoskote,NaN,SJowsn,800 - 2660,NaN,NaN,28.545,<NA>
7680,Plot Area,Immediate Possession,Kasavanhalli,NaN,NaN,5000,NaN,NaN,400.000,<NA>


In [28]:
df = df.dropna(subset=["bedrooms"])

In [29]:
# Handle missing bath

In [30]:
df["bath"].value_counts(dropna=False).sort_index()

bath
1.0      762
2.0     6546
3.0     3179
4.0     1198
5.0      520
6.0      271
7.0      102
8.0       64
9.0       42
10.0      13
11.0       3
12.0       7
13.0       3
14.0       1
15.0       1
16.0       2
18.0       1
27.0       1
40.0       1
NaN       57
Name: count, dtype: int64

In [31]:
df["bath"].median()

np.float64(2.0)

In [32]:
df["bath"] = df["bath"].fillna(df["bath"].median())

In [33]:
# Handle missing balcony

In [34]:
df["balcony"].value_counts(dropna=False).sort_index()

balcony
0.0    1010
1.0    4700
2.0    4845
3.0    1630
NaN     589
Name: count, dtype: int64

In [35]:
df["balcony"].median()

np.float64(2.0)

In [36]:
df["balcony"] = df["balcony"].fillna(df["balcony"].median())

In [37]:
# Handle missing society

In [38]:
df.drop(columns=["society"], inplace=True)

In [39]:
# Step 6: Clean the total_sqft column

In [40]:
df["total_sqft"].head(20)

0     1056
1     2600
2     1440
3     1521
4     1200
5     1170
6     2732
7     3300
8     1310
9     1020
10    1800
11    2785
12    1000
13    1100
14    2250
15    1175
16    1180
17    1540
18    2770
19    1100
Name: total_sqft, dtype: str

In [41]:
df["total_sqft"].dtype

<StringDtype(storage='python', na_value=nan)>

In [42]:
SQFT_PATTERN = re.compile(
    r"^\s*(\d+(?:\.\d+)?)\s*(?:-\s*(\d+(?:\.\d+)?))?\s*(.*?)\s*$"
)

UNIT_FACTORS = {
    "sq. meter": 10.7639,
    "sq. meters": 10.7639,
    "sq. yards": 9,
    "acres": 43560,
    "guntha": 1089,
    "perch": 272.25
}


def convert_sqft(value):
    if pd.isna(value):
        return np.nan

    match = SQFT_PATTERN.match(str(value))

    if not match:
        return np.nan

    start, end, unit = match.groups()

    # Convert ranges to their midpoint
    sqft = (
        (float(start) + float(end)) / 2
        if end else float(start)
    )

    unit = unit.strip().lower()

    # Convert supported units to square feet
    factor = UNIT_FACTORS.get(unit, 1)

    return sqft * factor

In [43]:
df["total_sqft"] = df["total_sqft"].apply(convert_sqft)

In [44]:
df["total_sqft"].head(20)

0     1056.0
1     2600.0
2     1440.0
3     1521.0
4     1200.0
5     1170.0
6     2732.0
7     3300.0
8     1310.0
9     1020.0
10    1800.0
11    2785.0
12    1000.0
13    1100.0
14    2250.0
15    1175.0
16    1180.0
17    1540.0
18    2770.0
19    1100.0
Name: total_sqft, dtype: float64

In [45]:
df["total_sqft"].dtype

dtype('float64')

In [46]:
# Step 7: Validate numerical columns

In [47]:
df[["bedrooms", "total_sqft", "bath", "balcony", "price"]].describe()

,bedrooms,total_sqft,bath,balcony,price
count,12774.0,1.277400e+04,12774.000000,12774.000000,12774.000000
mean,2.817833,1.884424e+03,2.705417,1.601534,114.339147
std,1.311892,1.667326e+04,1.355605,0.808135,151.506067
min,1.0,1.000000e+00,1.000000,0.000000,8.000000
25%,2.0,1.100000e+03,2.000000,1.000000,50.000000
50%,3.0,1.280000e+03,2.000000,2.000000,73.000000
75%,3.0,1.692000e+03,3.000000,2.000000,121.000000
max,43.0,1.306800e+06,40.000000,3.000000,3600.000000


In [48]:
(df[["bedrooms", "total_sqft", "bath", "price"]] <= 0).sum()

bedrooms      0
total_sqft    0
bath          0
price         0
dtype: Int64

In [49]:
df[
    (df["bedrooms"] <= 0) |
    (df["total_sqft"] <= 0) |
    (df["bath"] <= 0) |
    (df["price"] <= 0)
]

,area_type,availability,location,size,total_sqft,bath,balcony,price,bedrooms


In [50]:
# Step 8: Check and handle outliers

In [51]:
df["price"].describe(percentiles=[0.25, 0.50, 0.75, 0.95, 0.99])

count    12774.000000
mean       114.339147
std        151.506067
min          8.000000
25%         50.000000
50%         73.000000
75%        121.000000
95%        325.000000
99%        665.510000
max       3600.000000
Name: price, dtype: float64

In [52]:
df["total_sqft"].describe(percentiles=[0.25, 0.50, 0.75, 0.95, 0.99])

count    1.277400e+04
mean     1.884424e+03
std      1.667326e+04
min      1.000000e+00
25%      1.100000e+03
50%      1.280000e+03
75%      1.692000e+03
95%      3.352100e+03
99%      5.422810e+03
max      1.306800e+06
Name: total_sqft, dtype: float64

In [53]:
df["price_per_sqft"] = (
    df["price"] * 100000 / df["total_sqft"]
)

In [54]:
df["price_per_sqft"].describe(
    percentiles=[0.01, 0.25, 0.50, 0.75, 0.95, 0.99]
)

count    1.277400e+04
mean     8.594905e+03
std      1.185515e+05
min      2.257423e+00
1%       2.500000e+03
25%      4.296441e+03
50%      5.482380e+03
75%      7.404660e+03
95%      1.550436e+04
99%      2.427629e+04
max      1.200000e+07
Name: price_per_sqft, dtype: float64

In [55]:
df.nlargest(10, "price_per_sqft")[
    ["location", "bedrooms", "total_sqft", "price", "price_per_sqft"]
]

,location,bedrooms,total_sqft,price,price_per_sqft
4086,Sarjapur Road,4,1.0,120.0,1.200000e+07
9423,Ramamurthy Nagar,1,1.0,48.0,4.800000e+06
5453,Kannur,6,3.0,75.0,2.500000e+06
4972,Srirampuram,7,5.0,115.0,2.300000e+06
349,Suragajakkanahalli,3,11.0,74.0,6.727273e+05
1122,Grihalakshmi Layout,5,24.0,150.0,6.250000e+05
11558,Whitefield,4,60.0,218.0,3.633333e+05
1020,Weavers Colony,1,15.0,30.0,2.000000e+05
7657,Raghuvanahalli,1,425.0,750.0,1.764706e+05
5668,Judicial Layout,5,405.0,400.0,9.876543e+04


In [56]:
# Step 9: Validate categorical values

In [57]:
for col in ["area_type", "availability", "location"]:
    print(f"\n{col}:")
    print(df[col].nunique())


area_type:
4

availability:
80

location:
1293


In [58]:
df["location"].value_counts().head(20)

location
Whitefield                  523
Sarjapur Road               377
Electronic City             289
Kanakpura Road              249
Thanisandra                 232
Yelahanka                   209
Marathahalli                169
Hebbal                      161
Raja Rajeshwari Nagar       159
Uttarahalli                 157
Hennur Road                 149
Bannerghatta Road           149
7th Phase JP Nagar          135
Electronic City Phase II    128
Rajaji Nagar                107
Haralur Road                102
Bellandur                    93
KR Puram                     91
Electronics City Phase 1     88
Hoodi                        88
Name: count, dtype: int64

In [59]:
# Step 10: Convert columns to appropriate data types

In [60]:
numeric_columns = [
    "total_sqft",
    "bath",
    "balcony",
    "price",
    "price_per_sqft"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [61]:
df["bedrooms"] = pd.to_numeric(
    df["bedrooms"], errors="coerce"
).astype("Int64")

In [62]:
df.info()

<class 'pandas.DataFrame'>
Index: 12774 entries, 0 to 13318
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   area_type       12774 non-null  str    
 1   availability    12774 non-null  str    
 2   location        12774 non-null  str    
 3   size            12774 non-null  str    
 4   total_sqft      12774 non-null  float64
 5   bath            12774 non-null  float64
 6   balcony         12774 non-null  float64
 7   price           12774 non-null  float64
 8   bedrooms        12774 non-null  Int64  
 9   price_per_sqft  12774 non-null  float64
dtypes: Int64(1), float64(5), str(4)
memory usage: 1.1 MB


In [63]:
# Step 11: Final missing-value check

In [64]:
df.isnull().sum()

area_type         0
availability      0
location          0
size              0
total_sqft        0
bath              0
balcony           0
price             0
bedrooms          0
price_per_sqft    0
dtype: int64

In [65]:
df[df["total_sqft"].isnull()]

,area_type,availability,location,size,total_sqft,bath,balcony,price,bedrooms,price_per_sqft


In [66]:
# Step 12: Final duplicate check

In [67]:
df.duplicated().sum()

np.int64(41)

In [68]:
df = df.drop_duplicates()

In [69]:
df.shape

(12733, 10)

In [70]:
# Step 13: Review the cleaned dataset

In [71]:
df.head()

,area_type,availability,location,size,total_sqft,bath,balcony,price,bedrooms,price_per_sqft
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,1056.0,2.0,1.0,39.07,2,3699.810606
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,2600.0,5.0,3.0,120.00,4,4615.384615
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,1440.0,2.0,3.0,62.00,3,4305.555556
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,1521.0,3.0,1.0,95.00,3,6245.890861
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,1200.0,2.0,1.0,51.00,2,4250.000000


In [72]:
df.columns.tolist()

['area_type',
 'availability',
 'location',
 'size',
 'total_sqft',
 'bath',
 'balcony',
 'price',
 'bedrooms',
 'price_per_sqft']

In [73]:
df.dtypes

area_type             str
availability          str
location              str
size                  str
total_sqft        float64
bath              float64
balcony           float64
price             float64
bedrooms            Int64
price_per_sqft    float64
dtype: object

In [74]:
df.describe()

,total_sqft,bath,balcony,price,bedrooms,price_per_sqft
count,1.273300e+04,12733.000000,12733.000000,12733.000000,12733.0,1.273300e+04
mean,1.886527e+03,2.707296,1.601822,114.519202,2.819524,8.607742e+03
std,1.670004e+04,1.357254,0.808922,151.707663,1.313409,1.187420e+05
min,1.000000e+00,1.000000,0.000000,8.000000,1.0,2.257423e+00
25%,1.100000e+03,2.000000,1.000000,50.000000,2.0,4.298893e+03
50%,1.282000e+03,2.000000,2.000000,73.000000,3.0,5.489774e+03
75%,1.693000e+03,3.000000,2.000000,122.000000,3.0,7.407407e+03
max,1.306800e+06,40.000000,3.000000,3600.000000,43.0,1.200000e+07


In [75]:
df.info()

<class 'pandas.DataFrame'>
Index: 12733 entries, 0 to 13318
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   area_type       12733 non-null  str    
 1   availability    12733 non-null  str    
 2   location        12733 non-null  str    
 3   size            12733 non-null  str    
 4   total_sqft      12733 non-null  float64
 5   bath            12733 non-null  float64
 6   balcony         12733 non-null  float64
 7   price           12733 non-null  float64
 8   bedrooms        12733 non-null  Int64  
 9   price_per_sqft  12733 non-null  float64
dtypes: Int64(1), float64(5), str(4)
memory usage: 1.1 MB


In [76]:
# Step 14: Export the cleaned dataset

In [77]:
df.to_csv(
    "Bengaluru_House_Data_Cleaned.csv",
    index=False
)